# Evaluation Analysis

Deep-dive into evaluation results across all pipeline stages.

- Perplexity trend across checkpoints
- MMLU accuracy by subject
- Win rate breakdown by prompt category
- Generation quality inspection

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

EVAL_DIR = Path("/results/eval")

def load_latest_summary(stage: str) -> dict | None:
    """Load the most recent eval summary for a stage."""
    stage_dir = EVAL_DIR / stage
    if not stage_dir.exists():
        return None
    runs = sorted(stage_dir.glob("*/summary.json"))
    if not runs:
        return None
    with open(runs[-1]) as f:
        return json.load(f)

stages = ["pretrain", "sft", "dpo"]
summaries = {s: load_latest_summary(s) for s in stages}

for stage, summary in summaries.items():
    if summary:
        ppl = summary["metrics"].get("perplexity", {}).get("val_perplexity", "N/A")
        print(f"  {stage:<12} perplexity={ppl}")
    else:
        print(f"  {stage:<12} (no eval results yet)")

## Perplexity Across Stages

In [ ]:
ppl_data = {
    s: summaries[s]["metrics"]["perplexity"]["val_perplexity"]
    for s in stages
    if summaries.get(s) and "perplexity" in summaries[s].get("metrics", {})
}

if ppl_data:
    fig, ax = plt.subplots(figsize=(7, 4))
    colors = ["#378ADD", "#7F77DD", "#D85A30"]
    bars = ax.bar(list(ppl_data.keys()), list(ppl_data.values()),
                  color=colors[:len(ppl_data)], edgecolor="white", linewidth=0.5)
    ax.set_ylabel("Validation perplexity")
    ax.set_title("Perplexity by training stage")
    ax.axhline(25, color="gray", linestyle=":", linewidth=1, label="Random baseline (vocab~32k)")
    for bar, val in zip(bars, ppl_data.values()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f"{val:.1f}", ha="center", va="bottom", fontsize=10)
    ax.legend(fontsize=9)
    ax.set_ylim(0, max(ppl_data.values()) * 1.2)
    plt.tight_layout()
    plt.savefig("/results/eval/perplexity_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Note: perplexity increasing after SFT is expected — distribution shift from raw text.")
else:
    print("No perplexity data yet. Run: make eval-pretrain, make eval-sft, make eval-dpo")

## MMLU Results

In [ ]:
dpo_summary = summaries.get("dpo")
mmlu = dpo_summary["metrics"].get("mmlu") if dpo_summary else None

if mmlu:
    overall = mmlu["overall_accuracy"]
    by_subject = mmlu["by_subject"]

    print(f"Overall MMLU accuracy: {overall:.1%}")
    print(f"Random baseline:       25.0%")
    print(f"Above random by:       {(overall - 0.25)*100:.1f}pp")

    fig, ax = plt.subplots(figsize=(10, 5))
    subjects = list(by_subject.keys())
    accuracies = [by_subject[s] for s in subjects]
    colors = ["#5DCAA5" if a > 0.25 else "#F0997B" for a in accuracies]

    bars = ax.barh(subjects, accuracies, color=colors, edgecolor="white", linewidth=0.5)
    ax.axvline(0.25, color="#888780", linestyle="--", linewidth=1.2, label="Random baseline (25%)")
    ax.axvline(overall, color="#534AB7", linestyle="-", linewidth=1.5,
               label=f"Overall ({overall:.1%})")
    ax.set_xlabel("Accuracy")
    ax.set_title("MMLU accuracy by subject — DPO checkpoint")
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    ax.set_xlim(0, 1)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig("/results/eval/mmlu_by_subject.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No MMLU data yet. Run: make eval-dpo (with --n-mmlu 100)")

## Win Rate Analysis

In [ ]:
win_rate = dpo_summary["metrics"].get("win_rate") if dpo_summary else None

if win_rate:
    wr  = win_rate["win_rate"]
    tr  = win_rate["tie_rate"]
    lr  = win_rate["loss_rate"]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Pie chart
    axes[0].pie(
        [wr, tr, lr],
        labels=[f"DPO wins ({wr:.1%})", f"Tie ({tr:.1%})", f"SFT wins ({lr:.1%})"],
        colors=["#5DCAA5", "#D3D1C7", "#F0997B"],
        startangle=90,
        wedgeprops={"edgecolor": "white", "linewidth": 1.5},
    )
    axes[0].set_title(f"DPO vs SFT win rate\n(n={win_rate['n_pairs']} pairs)")

    # Sample comparisons
    samples = win_rate.get("samples", [])[:10]
    if samples:
        outcomes = [s["outcome"] for s in samples]
        outcome_counts = {"policy_wins": outcomes.count("policy_wins"),
                         "tie": outcomes.count("tie"),
                         "ref_wins": outcomes.count("ref_wins")}
        axes[1].bar(
            ["DPO wins", "Tie", "SFT wins"],
            [outcome_counts["policy_wins"], outcome_counts["tie"], outcome_counts["ref_wins"]],
            color=["#5DCAA5", "#D3D1C7", "#F0997B"],
            edgecolor="white"
        )
        axes[1].set_title("Sample breakdown (first 10 pairs)")
        axes[1].set_ylabel("Count")

    plt.suptitle("DPO alignment — win rate vs SFT reference", fontsize=12)
    plt.tight_layout()
    plt.savefig("/results/eval/win_rate.png", dpi=150, bbox_inches="tight")
    plt.show()

    print(f"\nJudge: {win_rate.get('judge', 'N/A')}")
    print(f"Note: {win_rate.get('note', '')}")
else:
    print("No win rate data yet. Run: make eval-dpo")

## Generation Sample Inspector

In [ ]:
from IPython.display import display, HTML

for stage in ["pretrain", "sft", "dpo"]:
    summary = summaries.get(stage)
    if not summary:
        continue
    gen = summary["metrics"].get("generation")
    if not gen:
        continue

    print(f"\n{'='*60}")
    print(f"Stage: {stage.upper()}")
    print('='*60)
    for sample in gen.get("samples", [])[:3]:
        print(f"\n[{sample['category']}]")
        print(f"Prompt:   {sample['prompt'][:100]}")
        print(f"Response: {sample['response'][:300]}")